# CNNs avec PyTorch Lightning sur des données de paysage

## Vérification de l'utilisation de GPU

Allez dans le menu `Exécution > Modifier le type d'execution` et vérifiez que l'on est bien en Python 3 et que l'accélérateur matériel est configuré sur « GPU ».

In [ ]:
!nvidia-smi

## Téléchargement du dataset Landscape depuis un repo git

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s landscape
!ls landscape
print("***")
!ls -l landscape/seg_train
print("***")
!ls -l landscape/seg_pred

## Installation et import de PyTorch Lightning et des autres librairies nécessaires

In [ ]:
!pip install -q lightning torchmetrics timm

In [ ]:
import itertools
import pathlib

import lightning
import matplotlib.pyplot as plt
import numpy
import pandas
import PIL.Image
import seaborn
import sklearn.metrics
import torch
import torchmetrics
import tqdm.notebook
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, Dataset

## Préparation des données

Pour charger nos données, nous allons combiner plusieurs libraires : Pillow, NumPy & PyTorch.

In [ ]:
INPUT_SHAPE = (3, 150, 150)


label_names = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
label_to_index = {l: i for i, l in enumerate(label_names)}


def get_images(dir_path: pathlib.Path,
               size: int = INPUT_SHAPE[1],
               shuffle: bool = True,
               create_labels: bool = True,
               ) -> tuple[torch.Tensor, torch.Tensor] | torch.Tensor:
  images = []
  if create_labels:
    labels = []

  # On itère sur les sous-dossier de la racine : ils correspondent chacun à une
  # classe
  for subdir_path in tqdm.notebook.tqdm(
      list(dir_path.iterdir()), desc="Traitement des dossiers"):

    dir_name = subdir_path.name

    if create_labels:
      # On attribue le bon label en fonction du nom du dossier "labels"
      label = label_to_index.get(dir_name)

    # On ajoute chaque image du label (dossier) courant à notre dataset
    for image_path in tqdm.notebook.tqdm(
        list(subdir_path.iterdir()), desc=f"Dossier {dir_name}", leave=False):
      # Utilisation de PIL pour charger l'image
      images.append(
          numpy.array(
              PIL.Image.open(image_path).convert("RGB").resize((size, size))))
      if create_labels:
        labels.append(label)

  # PyTorch attend les canaux en deuxième position : on passe donc de la forme
  # (N, H, W, C) à la forme (N, C, H, W)
  images = torch.from_numpy(numpy.array(images)).permute(0, 3, 1, 2).contiguous()
  if create_labels:
    labels = torch.tensor(labels)

  if shuffle:
    perm = torch.randperm(images.shape[0])
    images = images[perm]
    if create_labels:
      labels = labels[perm]

  if create_labels:
    return images, labels
  else:
    return images

In [ ]:
NUM_WORKERS = 2


class LandscapeDataset(Dataset):
  """Dataset d'images, converties en flottants dans [0, 1] à la volée."""

  def __init__(self,
               images: torch.Tensor,
               labels: torch.Tensor | None = None) -> None:
    self.images = images
    self.labels = labels

  def __len__(self) -> int:
    return self.images.shape[0]

  def __getitem__(
      self,
      index: int) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
    image = self.images[index].float() / 255.
    if self.labels is None:
      return image
    return image, self.labels[index]


def make_loader(images: torch.Tensor,
                labels: torch.Tensor | None = None,
                batch_size: int = 32,
                shuffle: bool = False) -> DataLoader:
  return DataLoader(LandscapeDataset(images, labels),
                    batch_size=batch_size,
                    shuffle=shuffle,
                    num_workers=NUM_WORKERS,
                    persistent_workers=NUM_WORKERS > 0)


def get_dataloaders(images: torch.Tensor,
                    labels: torch.Tensor,
                    batch_size: int = 32,
                    validation_split: float = 0.3
                    ) -> tuple[DataLoader, DataLoader]:
  """Sépare les données entre entraînement et validation."""
  n_validation = int(images.shape[0] * validation_split)
  n_train = images.shape[0] - n_validation
  return (make_loader(images[:n_train],
                      labels[:n_train],
                      batch_size,
                      shuffle=True),
          make_loader(images[n_train:], labels[n_train:], batch_size))


def to_displayable(image: torch.Tensor) -> numpy.ndarray:
  """Passe d'un tenseur (C, H, W) à un tableau (H, W, C) affichable."""
  return image.permute(1, 2, 0).numpy()

## Appel à `get_images`

In [ ]:
images, labels = get_images(pathlib.Path("landscape") / "seg_train")

In [ ]:
print(f"Forme des images : {tuple(images.shape)}")
print(f"Forme des labels : {tuple(labels.shape)}")

seaborn.countplot(x=labels.numpy())
plt.title("Décomptes des différents labels")
plt.ylabel("Décompte")
plt.xlabel("Label")
plt.show()

In [ ]:
# Création de la grille de sous-plots. On donne l'argument figsize pour agrandir
# la taille de la figure qui est petite par défaut
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# On choisit 25 indices au hasard, sans replacement (on ne veut pas afficher la
# même image deux fois)
random_indexes = numpy.random.choice(images.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    image = images[img_index]
    label = label_names[int(labels[img_index])]

    # Affichage avec matplotlib et sa fonction imshow, très pratique en vision par
    # ordinateur
    ax[i, j].imshow(to_displayable(image))
    ax[i, j].set_title(f"Exemple {img_index} ({label})")
    ax[i, j].axis('off')

## Création du modèle

Voici un exemple de CNN « minimaliste »

In [ ]:
class ImageClassifier(lightning.LightningModule):
  """Enveloppe Lightning commune à tous nos modèles de classification.

  Le modèle encapsulé renvoie des logits : le softmax est appliqué par la
  fonction de perte pendant l'apprentissage, et explicitement au moment de la
  prédiction.
  """

  def __init__(self,
               model: nn.Module,
               learning_rate: float = 1e-4,
               weight_decay: float = 0.,
               num_classes: int = len(label_names),
               input_size: int = INPUT_SHAPE[1],
               ) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, INPUT_SHAPE[0], input_size,
                                           input_size)
    # torchmetrics demande une instance de métrique par étape
    self.accuracies = nn.ModuleDict({
        f"{stage}_accuracy": torchmetrics.Accuracy(task="multiclass",
                                                   num_classes=num_classes)
        for stage in ("train", "val", "test")
    })

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor],
            stage: str) -> torch.Tensor:
    images, labels = batch
    logits = self(images)
    loss = nn.functional.cross_entropy(logits, labels)
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, labels)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True,
             prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True,
             prog_bar=True)
    return loss

  def training_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                batch_index: int) -> torch.Tensor:
    return self._step(batch, "test")

  def predict_step(self, batch: torch.Tensor | tuple[torch.Tensor, ...],
                   batch_index: int) -> torch.Tensor:
    images = batch[0] if isinstance(batch, (list, tuple)) else batch
    return self(images).softmax(dim=-1).cpu()

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(),
                            lr=self.hparams.learning_rate,
                            weight_decay=self.hparams.weight_decay)


def make_trainer(max_epochs: int, name: str) -> lightning.Trainer:
  """Crée un Trainer qui journalise ses métriques dans un fichier CSV."""
  return lightning.Trainer(max_epochs=max_epochs,
                           accelerator="auto",
                           devices=1,
                           logger=CSVLogger("logs", name=name),
                           enable_checkpointing=False,
                           num_sanity_val_steps=0,
                           log_every_n_steps=10)

In [ ]:
# Initialisation et définition du modéle

# Le modèle est un empilement de couches où le flux de données est séquentiel
model = nn.Sequential(
    # Une première couche de neurones de 1 convolutions de 3x3 pixels, prenant
    # en entrée les 3 canaux de l'image, suivie de son activation
    nn.Conv2d(3, 1, kernel_size=3),
    nn.ReLU(),
    # Une couche de max pooling
    nn.MaxPool2d(3, 3),
    # Une couche de manipulation des tenseurs : suppression de toutes les
    # dimensions sauf celle de batch et une autre qui contient toutes les valeurs
    nn.Flatten(),
    # Une couche de sortie dense avec 6 neurones (le softmax est appliqué par la
    # fonction de perte)
    nn.Linear(49 * 49, 6),
)

# Encapsulation du modèle : le LightningModule définit la fonction de perte et
# l'optimiseur
classifier = ImageClassifier(model, learning_rate=0.0001)

# Affichage d'un résumé du modèle
print(ModelSummary(classifier, max_depth=-1))

## Pouvez-vous expliquer les différents nombres de paramètres ?

### Solution


Premier layer de 1 convolutions : (taille du kernel) * (nb kernel) * (nb canaux en entrée) + (nb biais (= nb kernel)) = (3 * 3 ) * 1 * 3 + 1

Dernier layer dense : (input dim) * (output dim) + (nb biais) = 2401 * 6 + 6



## Apprentissage

Apprenons ce modèle sur nos données ! Dans un premier temps, nous entraînons sur une seule epoch pour simplement vérifier que notre modèle est opérationnel.

In [ ]:
# Apprentissage du modèle
train_loader, validation_loader = get_dataloaders(images, labels)
trainer = make_trainer(max_epochs=1, name="cnn_minimaliste")
trainer.fit(classifier, train_loader, validation_loader)

## Améliorez cette performance

Inspirez-vous du modèle précédent en rajoutant des couches, en faisant des couches plus petites ou plus grosses.

Visez entre 10 et 20 itérations et mois de 1 minute par itération (pour des raisons évidentes).

On peut considérer l'utilisation d'une couche de dropout juste avant la dernière couche dense pour améliorer la régularisation.

On peut obtenir une précision supérieure à 70% sur la base de validation en un temps raisonnable.

La solution proposée prend $\approx$ 45 secondes par itération pendant 15 itérations et atteint aux alentour de 85% d'accuracy sur la base de validation.

In [ ]:
# Vos améliorations ici
model = nn.Sequential(
    nn.Conv2d(3, 10, kernel_size=3),
    nn.ReLU(),
    nn.MaxPool2d(3, 3),
    nn.Flatten(),
    nn.Linear(10 * 49 * 49, 6),
)
classifier = ImageClassifier(model, learning_rate=0.0001)

# Affichage d'un résumé du modèle
print(ModelSummary(classifier, max_depth=-1))

# Apprentissage du modèle
train_loader, validation_loader = get_dataloaders(images, labels)
trainer = make_trainer(max_epochs=10, name="ameliorations")
trainer.fit(classifier, train_loader, validation_loader)


# Plot des métriques d'entraînement
def plot_metrics(trainer: lightning.Trainer) -> None:
  # Les métriques d'entraînement et de validation ne sont pas écrites sur les
  # mêmes lignes du fichier CSV : on les regroupe par epoch
  metrics = pandas.read_csv(
      pathlib.Path(trainer.logger.log_dir) / "metrics.csv").groupby(
          "epoch").mean(numeric_only=True).dropna(
              subset=["train_loss", "val_loss"])

  plt.plot(metrics.index, metrics["train_accuracy"])
  plt.plot(metrics.index, metrics["val_accuracy"])
  plt.title("Accuracy du modèle")
  plt.ylabel("Accuracy")
  plt.xlabel("Epoch")
  plt.legend(["Entraînement", "Validation"], loc="upper left")
  plt.show()

  plt.plot(metrics.index, metrics["train_loss"])
  plt.plot(metrics.index, metrics["val_loss"])
  plt.title("Perte du modèle")
  plt.ylabel("Perte")
  plt.xlabel("Epoch")
  plt.legend(["Entraînement", "Validation"], loc="upper right")
  plt.show()


plot_metrics(trainer)

### Solution

In [ ]:
def conv(in_channels: int, out_channels: int = 200) -> nn.Sequential:
  convolution = nn.Conv2d(in_channels,
                          out_channels,
                          kernel_size=3,
                          padding="same")
  nn.init.orthogonal_(convolution.weight)
  return nn.Sequential(convolution, nn.ReLU())


def pooling() -> nn.MaxPool2d:
  return nn.MaxPool2d(2, 2, ceil_mode=True)


def dropout(rate: float = 0.2) -> nn.Dropout:
  return nn.Dropout(rate)


def dense(in_features: int, out_features: int) -> nn.Sequential:
  return nn.Sequential(nn.Linear(in_features, out_features), nn.ReLU())


model = nn.Sequential(
    conv(3),
    pooling(),
    conv(200),
    pooling(),
    conv(200),
    pooling(),
    conv(200),
    pooling(),
    conv(200),
    pooling(),
    conv(200),
    pooling(),
    conv(200),
    nn.Flatten(),
    dropout(),
    dense(200 * 3 * 3, 200),
    dropout(),
    dense(200, 100),
    dropout(),
    dense(100, 50),
    dropout(),
    nn.Linear(50, 6),
)

classifier = ImageClassifier(model, learning_rate=0.0001)

# Affichage d'un résumé du modèle
print(ModelSummary(classifier, max_depth=-1))

In [ ]:
# Apprentissage du modèle
train_loader, validation_loader = get_dataloaders(images,
                                                  labels,
                                                  batch_size=128)
trainer = make_trainer(max_epochs=15, name="cnn_profond")
trainer.fit(classifier, train_loader, validation_loader)

# Visualisation des métriques d'entrainement
plot_metrics(trainer)

Une implémentation de [LeNet](https://en.wikipedia.org/wiki/LeNet) :

In [ ]:
def conv(in_channels: int, out_channels: int, padding: str) -> nn.Sequential:
  return nn.Sequential(
      nn.Conv2d(in_channels, out_channels, kernel_size=5, padding=padding),
      nn.Sigmoid())


def pooling() -> nn.AvgPool2d:
  return nn.AvgPool2d(2)


def dense(in_features: int, out_features: int) -> nn.Sequential:
  return nn.Sequential(nn.Linear(in_features, out_features), nn.Sigmoid())


le_net = nn.Sequential(
    conv(3, 6, "same"),
    pooling(),
    conv(6, 16, "valid"),
    pooling(),
    nn.Flatten(),
    dense(16 * 35 * 35, 120),
    dense(120, 84),
    nn.Linear(84, 6),
)
le_net_classifier = ImageClassifier(le_net, learning_rate=1e-4)
print(ModelSummary(le_net_classifier, max_depth=-1))

train_loader, validation_loader = get_dataloaders(images,
                                                  labels,
                                                  batch_size=128)
le_net_trainer = make_trainer(max_epochs=15, name="le_net")
le_net_trainer.fit(le_net_classifier, train_loader, validation_loader)
plot_metrics(le_net_trainer)

Une implémentation d'[AlexNet](https://en.wikipedia.org/wiki/AlexNet) :

In [ ]:
def conv(in_channels: int,
         out_channels: int,
         kernel_size: int,
         padding: str = "same",
         stride: int = 1) -> nn.Sequential:
  return nn.Sequential(
      nn.Conv2d(in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding), nn.ReLU())


def pooling() -> nn.MaxPool2d:
  return nn.MaxPool2d(kernel_size=3, stride=2)


def dropout(rate: float = 0.5) -> nn.Dropout:
  return nn.Dropout(rate)


def dense(in_features: int, out_features: int) -> nn.Sequential:
  return nn.Sequential(nn.Linear(in_features, out_features), nn.ReLU())


alex_net = nn.Sequential(
    conv(3, 96, 11, "valid", 4),
    pooling(),
    conv(96, 256, 5),
    pooling(),
    conv(256, 384, 3),
    conv(384, 384, 3),
    conv(384, 256, 3),
    pooling(),
    nn.Flatten(),
    dense(256 * 3 * 3, 4096),
    dropout(),
    dense(4096, 4096),
    dropout(),
    nn.Linear(4096, 6),
)
alex_net_classifier = ImageClassifier(alex_net, learning_rate=1e-4)
print(ModelSummary(alex_net_classifier, max_depth=-1))

train_loader, validation_loader = get_dataloaders(images,
                                                  labels,
                                                  batch_size=128)
alex_net_trainer = make_trainer(max_epochs=15, name="alex_net")
alex_net_trainer.fit(alex_net_classifier, train_loader, validation_loader)
plot_metrics(alex_net_trainer)

Une implémentation de bloc [Inception](https://towardsdatascience.com/a-simple-guide-to-the-versions-of-the-inception-network-7fc52b863202) (pour constituer un réseau complet il faudrait en agencer plusieurs). Cette implémentation définit un [module PyTorch](https://docs.pytorch.org/docs/stable/notes/modules.html) personnalisé, car le flux de données n'est plus séquentiel :

In [ ]:
class InceptionBlock(nn.Module):
  """Bloc Inception : quatre branches concaténées sur la dimension des canaux."""

  def __init__(self, in_channels: int, out_channels: int = 64) -> None:
    super().__init__()

    # conv 1x1
    self.conv11 = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 1), nn.ReLU())

    # conv 1x1 puis conv 3x3
    self.conv33 = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 1), nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, 3, padding="same"), nn.ReLU())

    # conv 1x1 puis conv 5x5
    self.conv55 = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 1), nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, 5, padding="same"), nn.ReLU())

    # max pool 3x3 strides 1x1 puis conv 1x1
    self.max_pool = nn.Sequential(
        nn.MaxPool2d(3, 1, padding=1),
        nn.Conv2d(in_channels, out_channels, 1), nn.ReLU())

  def forward(self, inputs: torch.Tensor) -> torch.Tensor:
    # concaténation
    return torch.cat([
        self.conv11(inputs),
        self.conv33(inputs),
        self.conv55(inputs),
        self.max_pool(inputs),
    ], dim=1)


def inception() -> ImageClassifier:
  model = nn.Sequential(
      InceptionBlock(INPUT_SHAPE[0]),
      nn.Flatten(),
      nn.Linear(4 * 64 * 150 * 150, 6),
  )
  classifier = ImageClassifier(model, learning_rate=0.0001)
  print(ModelSummary(classifier, max_depth=-1))
  return classifier


inception_classifier = inception()
train_loader, validation_loader = get_dataloaders(images, labels)
inception_trainer = make_trainer(max_epochs=10, name="inception")
inception_trainer.fit(inception_classifier, train_loader, validation_loader)

## Évaluation des performances sur l'ensemble de test

Dans le dossier `seg_test` se trouve un ensemble de données qui n'ont jamais été vues durant l'apprentissage.

On utilisera la méthode `test(model, dataloaders)` du `Trainer` pour évaluer la qualité de nos prédictions sur ce dataset.

In [ ]:
test_images, test_labels = get_images(
    pathlib.Path("landscape") / "seg_test")
test_loader = make_loader(test_images, test_labels, batch_size=128)

# Le Trainer qui a entraîné le modèle est aussi celui qui l'évalue
trainer.test(classifier, dataloaders=test_loader)


def predict_probabilities(trainer: lightning.Trainer,
                          classifier: ImageClassifier,
                          images: torch.Tensor,
                          batch_size: int = 128) -> torch.Tensor:
  """Renvoie la probabilité attribuée à chaque classe pour chaque image."""
  return torch.cat(
      trainer.predict(classifier, make_loader(images, batch_size=batch_size)))

## Analyse d'erreur

On affiche la matrice de confusion puis on regarde des images mal classées.

In [ ]:
def analyze_preds(preds: torch.Tensor, labels: torch.Tensor) -> None:
  confusion_matrix = sklearn.metrics.confusion_matrix(labels, preds)
  seaborn.heatmap(confusion_matrix,
                  annot=True,
                  fmt="d",
                  cmap="rocket_r",
                  xticklabels=label_names,
                  yticklabels=label_names)
  plt.title("Matrice de confusion")
  plt.show()

  seaborn.countplot(x=list(map(lambda x: label_names[x], preds.tolist())))
  plt.title("Décomptes des classes prédites")
  plt.ylabel("Décompte")
  plt.xlabel("Class")
  plt.show()


test_pred = predict_probabilities(trainer, classifier,
                                  test_images).argmax(dim=-1)
analyze_preds(test_pred, test_labels)

In [ ]:
def plot_mistakes(predicted_class: str, true_class: str) -> None:
  print(f"Prédiction : {predicted_class}, classe réelle : {true_class}")
  mistakes = test_images[(test_pred == label_names.index(predicted_class))
                         & (test_labels == label_names.index(true_class))]
  random_indexes = numpy.random.choice(mistakes.shape[0],
                                       size=min(mistakes.shape[0], 25),
                                       replace=False)
  grid_indexes = itertools.product(range(5), repeat=2)

  _, ax = plt.subplots(5, 5, figsize=(15, 15))
  for i, j in itertools.product(range(5), repeat=2):
    ax[i, j].axis("off")
  for img_index, (i, j) in zip(random_indexes, grid_indexes):
    ax[i, j].imshow(to_displayable(mistakes[img_index]))
  plt.show()

In [ ]:
# Plot les images prédites glacier alors qu'elles ont un label montagne
plot_mistakes("glacier", "mountain")

In [ ]:
# Plot les images prédites glacier alors qu'elles ont un label mer
plot_mistakes("glacier", "sea")

In [ ]:
# Plot les images prédites bâtiment alors qu'elles ont un label mer
plot_mistakes("buildings", "sea")

## Transfert d'apprentissage

In [ ]:
import timm


class Normalize(nn.Module):
  """Normalise, canal par canal, des images déjà ramenées dans [0, 1]."""

  def __init__(self, mean: tuple[float, ...] | float,
               std: tuple[float, ...] | float) -> None:
    super().__init__()
    self.register_buffer("mean", torch.tensor(mean).reshape(1, -1, 1, 1))
    self.register_buffer("std", torch.tensor(std).reshape(1, -1, 1, 1))

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return (images - self.mean) / self.std


class Frozen(nn.Module):
  """Encapsule un module gelé, maintenu en mode évaluation."""

  def __init__(self, module: nn.Module) -> None:
    super().__init__()
    self.module = module.eval().requires_grad_(False)

  def train(self, mode: bool = True) -> "Frozen":
    return super().train(False)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.module(images)


base_model = timm.create_model("tf_efficientnetv2_b0.in1k",
                               pretrained=True,
                               num_classes=0,
                               global_pool="")
data_config = timm.data.resolve_model_data_config(base_model)

model = nn.Sequential(
    Normalize(data_config["mean"], data_config["std"]),
    Frozen(base_model),
    nn.Dropout2d(0.1),
    nn.Conv2d(base_model.num_features, 1024, 1),
    nn.ReLU(),
    nn.Dropout2d(0.1),
    nn.Conv2d(1024, 256, 1),
    nn.ReLU(),
    nn.Dropout2d(0.1),
    nn.Conv2d(256, 64, 1),
    nn.ReLU(),
    nn.Dropout2d(0.1),
    nn.Conv2d(64, 16, 1),
    nn.ReLU(),
    nn.Dropout2d(0.1),
    nn.Flatten(),
    # Les cartes de caractéristiques du réseau pré-entraîné font 5x5 pixels
    nn.Linear(16 * 5 * 5, 6),
)

# La régularisation L2 de la dernière couche devient une pénalité sur les poids
# de l'optimiseur : seuls les poids de la tête de classification sont concernés,
# ceux du réseau pré-entraîné étant gelés
classifier = ImageClassifier(model, learning_rate=0.0001, weight_decay=0.01)

print(ModelSummary(classifier, max_depth=-1))

In [ ]:
train_loader, validation_loader = get_dataloaders(images, labels, batch_size=64)
trainer = make_trainer(max_epochs=15, name="transfert")
trainer.fit(classifier, train_loader, validation_loader)

plot_metrics(trainer)

In [ ]:
trainer.test(classifier, dataloaders=test_loader)
test_pred = predict_probabilities(trainer, classifier,
                                  test_images).argmax(dim=-1)
analyze_preds(test_pred, test_labels)
plot_mistakes("glacier", "mountain")
plot_mistakes("glacier", "sea")
plot_mistakes("buildings", "sea")

## Prédire dans des condition « réelles »

Dans le dossier `seg_pred` se trouvent des images non-annotées. On ne peut donc pas évaluer correctement les performances sur cet ensemble.

Cependant, on peut afficher des photos et les probabilités que notre modèle attribue à chaque classe.

In [ ]:
pred_images = get_images(
    pathlib.Path("landscape") / "seg_pred",
    create_labels=False)
pred_images.shape

In [ ]:
# Création de la grille de sous-plots. On donne l'argument figsize pour agrandir
# la taille de la figure qui est petite par défaut
_, ax = plt.subplots(10, 5, figsize=(30, 45))

# On choisit 25 indices au hasard, sans replacement (on ne veut pas afficher la
# même image deux fois)
random_indexes = numpy.random.choice(pred_images.shape[0],
                                     size=(5, 5),
                                     replace=False)

# Prédiction des classes des 25 images sélectionnées
probabilities = predict_probabilities(
    trainer, classifier, pred_images[random_indexes.ravel()]).reshape(5, 5, -1)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    # Récupération de l'image et de la classe prédite
    image = pred_images[img_index]
    predicted_class = label_names[int(probabilities[i, j].argmax())]

    # Affichage avec matplotlib et sa fonction imshow, très pratique en vision
    # par ordinateur
    ax[i * 2, j].imshow(to_displayable(image))
    ax[i * 2, j].set_title(f"Exemple {img_index} ({predicted_class})")
    ax[i * 2, j].axis('off')

    # Affichage de la distribution de prédiction sur la ligne d'en dessous
    ax[i * 2 + 1, j].bar(label_names, probabilities[i, j].numpy())

## Justesse en fonction de la probabilité maximale

Regardons maintenant si la justesse (*accuracy* en anglais) varie significativement en fonction de la probabilité maximale rendue par le modèle.

In [ ]:
test_probabilities = predict_probabilities(trainer, classifier, test_images)

In [ ]:
def threshold_accuracy(probabilities: torch.Tensor,
                       labels: torch.Tensor,
                       threshold: float
                       ) -> tuple[float, float, float, float]:
  predictions = probabilities.argmax(dim=-1)
  mask_above = probabilities.max(dim=-1).values > threshold
  mask_below = ~mask_above

  n_above = int(mask_above.sum())
  n_below = int(mask_below.sum())

  if n_above:
    above = float(
        (predictions[mask_above] == labels[mask_above]).sum()) / n_above
  else:
    above = 1.
  if n_below:
    below = float(
        (predictions[mask_below] == labels[mask_below]).sum()) / n_below
  else:
    below = 1.
  return above, below, n_above / labels.shape[0], n_below / labels.shape[0]


accuracy_above, accuracy_below, ratio_above, ratio_below = threshold_accuracy(
    test_probabilities, test_labels, 0.99)
print("Justesse pour les prédictions du modèle qui ont une probabilité au "
      f"dessus du seuil ({ratio_above * 100:.2f}% des données) : "
      f"{accuracy_above:.2f}")
print("Justesse pour les prédictions du modèle qui ont une probabilité en "
      f"dessous du seuil ({ratio_below * 100:.2f}% des données) : "
      f"{accuracy_below:.2f}")

xs = numpy.arange(101) / 100
accuracies = []
ratios_above = []
for x in xs:
  accuracy, _, ratio_above, _ = threshold_accuracy(test_probabilities,
                                                   test_labels,
                                                   x)
  accuracies.append(accuracy)
  ratios_above.append(ratio_above)
plt.plot(xs, accuracies, label="Justesse")
plt.plot(xs, ratios_above, label="Rappel")
plt.xlabel("Seuil de prédiction")
plt.title("Justesse et rappel en fonction du seuil de prédiction")
plt.legend()
plt.show()

## Transfert d'apprentissage avec des transformeurs

In [ ]:
images, labels = get_images(pathlib.Path("landscape") / "seg_train",
                            size=224)

In [ ]:
import transformers


class ViTFeatures(nn.Module):
  """Renvoie l'embedding du token de classification calculé par le ViT."""

  def __init__(self, vit: nn.Module) -> None:
    super().__init__()
    self.vit = vit

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.vit(pixel_values=images).last_hidden_state[:, 0]


vit = transformers.AutoModel.from_pretrained(
    "google/vit-base-patch16-224-in21k")

transferred_transformer = nn.Sequential(
    Normalize(0.5, 0.5),
    Frozen(ViTFeatures(vit)),
    nn.Linear(vit.config.hidden_size, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 6),
)
classifier = ImageClassifier(transferred_transformer,
                             learning_rate=0.001,
                             input_size=224)
print(ModelSummary(classifier, max_depth=-1))

In [ ]:
train_loader, validation_loader = get_dataloaders(images, labels, batch_size=64)
trainer = make_trainer(max_epochs=15, name="transferred_transformer")
trainer.fit(classifier, train_loader, validation_loader)

In [ ]:
test_images, test_labels = get_images(
    pathlib.Path("landscape") / "seg_test",
    size=224)